# Can we predict `passed_quiz`?
Protocol: `protocol.yaml`. Group holdout on `student_key` (`holdout.csv`), 5-fold GroupKFold CV on training rows, selection by mean CV ROC AUC, holdout scored once.

**Feature timing.** The quiz closes the session, so `session_minutes` (length of the whole session) is only final once the quiz is over: it is recorded at/after the outcome and is excluded. `hints_used` accrues during the session before the closing quiz, so it is kept (assumption: predictions are made when the quiz starts). `student_key` is an entity key and is excluded per the protocol.

In [ ]:
import json, hashlib
import numpy as np, pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.base import clone

assert hashlib.sha256(open('data.csv','rb').read()).hexdigest()=='5a0b0221ed641db40322e1a95cae32cc67d3525ecb500813f95d6c109b2a7409'
assert hashlib.sha256(open('holdout.csv','rb').read()).hexdigest()=='ff08c7a88fd7e15b17b05a390393550fdf57ea2b6df584f1d0594cc2ec1f00e6'
df = pd.read_csv('data.csv')
hold_idx = pd.read_csv('holdout.csv')['row'].to_numpy()
assert len(df)==12453 and len(hold_idx)==2465
train_idx = np.setdiff1d(np.arange(len(df)), hold_idx)
TARGET='passed_quiz'
FEATURES=['enrol_year','school_code','prior_score','hints_used']
X, y, g = df[FEATURES], df[TARGET], df['student_key']
assert not set(g.iloc[train_idx]) & set(g.iloc[hold_idx])
len(train_idx), y.iloc[train_idx].mean()

## Candidates (named before any scoring)

In [ ]:
candidates = {
    'baseline_prior': DummyClassifier(strategy='prior'),
    'logreg': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    'logreg_C0.1': make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=1000)),
    'random_forest': RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=0),
    'hist_gb': HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=0),
}
list(candidates)

## Cross-validation on training rows (preprocessing is inside the pipelines, so it is fit per fold)

In [ ]:
Xtr, ytr, gtr = X.iloc[train_idx], y.iloc[train_idx], g.iloc[train_idx]
folds = list(GroupKFold(n_splits=5).split(Xtr, ytr, gtr))
cv_folds = [[train_idx[a].tolist(), train_idx[b].tolist()] for a,b in folds]  # [fit rows, validation rows]
cv_scores = {}
for name, m in candidates.items():
    s=[]
    for a,b in folds:
        mm=clone(m).fit(Xtr.iloc[a], ytr.iloc[a])
        s.append(float(roc_auc_score(ytr.iloc[b], mm.predict_proba(Xtr.iloc[b])[:,1])))
    cv_scores[name]=s
cv = pd.DataFrame({k:{'mean':np.mean(v),'std':np.std(v)} for k,v in cv_scores.items()}).T
cv

## Choose by mean CV ROC AUC, refit on all training rows, score holdout once

In [ ]:
chosen = max(cv_scores, key=lambda k: np.mean(cv_scores[k]))
final = clone(candidates[chosen]).fit(Xtr, ytr)
holdout_auc = float(roc_auc_score(y.iloc[hold_idx], final.predict_proba(X.iloc[hold_idx])[:,1]))
print(chosen, 'CV', np.mean(cv_scores[chosen]), 'holdout ROC AUC', holdout_auc)

In [ ]:
evidence = {
    'features': FEATURES,
    'candidates': list(candidates),
    'cv_folds': cv_folds,
    'cv_scores': cv_scores,
    'chosen': chosen,
    'holdout_rows': hold_idx.tolist(),
    'holdout_scores': {chosen: holdout_auc},
}
json.dump(evidence, open('evidence.json','w'))

## Conclusion
The headline is the holdout ROC AUC printed above for the CV-chosen candidate; 0.5 is chance. Caveat: if predictions must be made before the session starts, `hints_used` would also have to go.